In [355]:
from typing import TypedDict, List, Optional, Dict, Any, Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.agents import Tool
import json
import os

from langchain_openai import ChatOpenAI

In [356]:
load_dotenv(override=True)

True

In [357]:
SYSTEM_MAX_ITERATIONS = 5

In [ ]:
class ContentState(TypedDict):
    topic: str
    target_audience: str
    primary_keyword: str
    search_intent: str
    desired_tone: str
    article: Optional[str]
    seo_evaluation: Optional[Dict[str, Any]]
    tone_evaluation: Optional[Dict[str, Any]]
    final_decision: Optional[Dict[str, Any]]
    iteration_count: int
    max_iterations: int
    strict_mode_level: int 


In [359]:
graph_builder = StateGraph(ContentState)

In [360]:
import nest_asyncio
nest_asyncio.apply()

In [361]:
STRATEGY_PROMPT = """
You are a content strategist. Your task is to analyze the given article topic and generate the following details:

1. **Target Audience**: Who is the primary reader of this article? Be specific.
2. **Primary Keyword**: The main SEO keyword that best represents the topic.
3. **Search Intent**: What type of searchers are looking for this topic? (Informational, Navigational, Transactional, or Commercial)
4. **Recommended Tone**: The tone of voice that will best engage the audience (e.g., professional, casual, friendly, authoritative, persuasive).

Return your output in a strict JSON format as follows:

{
  "target_audience": "...",
  "primary_keyword": "...",
  "search_intent": "...",
  "recommended_tone": "..."
}

**Topic:** 
"""


WRITER_PROMPT = """
You are a professional content writer.

Your task is to write a clear, engaging, and inclusive article based on the provided topic and requirements.

Instructions:
- Use clear, accessible language suitable for a global audience.
- Avoid cultural bias, stereotypes, or exclusionary language.
- Structure the article using:
  - One H1 title
  - Logical H2 and H3 subheadings
  - Short paragraphs (2–4 sentences max)
  - Bullet points where helpful
- Naturally incorporate the provided primary keyword and relevant related terms.
- Ensure the article satisfies the intended search intent (informational, transactional, etc.).
- Maintain consistent tone as specified.
- Do not explain your reasoning.
- Output only the final article in clean markdown format.

Input you will receive:
- Topic
- Target audience
- Primary keyword
- Search intent
- Desired tone
- Optional feedback from evaluators (if revision round)
"""

SEO_PROMPT = """
You are an SEO content evaluator.

Your task is to critically assess the article for search engine optimization quality.

Evaluate strictly on:

1. Keyword Optimization
   - Primary keyword presence (title, intro, headings)
   - Natural usage (no keyword stuffing)
   - Use of semantically related keywords

2. Structure & Formatting
   - Proper H1, H2, H3 hierarchy
   - Short paragraphs
   - Readability and scannability

3. Search Intent Alignment
   - Does the content satisfy the declared search intent?
   - Is it comprehensive enough?

4. Readability
   - Clear and accessible language
   - Minimal jargon

Return ONLY valid JSON in this format:

{
  "seo_score": number (0-100),
  "keyword_optimization_score": number (0-100),
  "structure_score": number (0-100),
  "search_intent_alignment_score": number (0-100),
  "readability_score": number (0-100),
  "issues": ["list specific issues"],
  "recommendation": "clear actionable improvement summary",
  "approved": true/false
}

Approval rule:
- approved = true only if seo_score >= 80
- Otherwise false

"""

TONE_PROMPT = """
You are a tone and audience alignment evaluator.

Your task is to assess whether the article maintains consistent tone, inclusivity, and audience alignment.

Evaluate strictly on:

1. Tone Consistency
   - Is the tone stable throughout?
   - Does it match the requested tone?

2. Audience Alignment
   - Is the language appropriate for the target audience?
   - Is it inclusive and culturally neutral?
   - Is jargon explained when necessary?

3. Engagement
   - Is the introduction compelling?
   - Does the content maintain reader interest?
   - Is the conclusion strong and clear?

4. Inclusivity
   - Avoids stereotypes or biased assumptions
   - Uses globally understandable examples

Return ONLY valid JSON in this format:

{
  "tone_score": number (0-100),
  "consistency_score": number (0-100),
  "audience_alignment_score": number (0-100),
  "engagement_score": number (0-100),
  "inclusivity_score": number (0-100),
  "issues": ["list specific issues"],
  "recommendation": "clear actionable improvement summary",
  "approved": true/false
}

Approval rule:
- approved = true only if tone_score >= 80
- Otherwise false


"""

EVALUATOR_PROMPT = """
You are a content quality decision agent.

You will receive:
- SEO evaluator JSON
- Tone evaluator JSON

Your job is to:

1. Determine final publish decision.
2. If both approved = true → approve for publishing.
3. If either approved = false → require revision.
4. Summarize the most critical improvement points.
5. Do not invent new feedback.
6. Base decision strictly on evaluator scores and issues.

Return ONLY valid JSON:

{
  "final_decision": "publish" or "revise",
  "reason_summary": "short explanation",
  "priority_fixes": ["top issues to address"],
  "needs_revision": true/false
}

Rule:
- publish only if BOTH evaluators approved = true
- otherwise revise
"""

In [362]:
from tools import other_tools
all_tools = await other_tools()

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(all_tools)

In [363]:
def strategy_planner_node(state: ContentState):
    topic = state["topic"]

    result = llm.invoke(STRATEGY_PROMPT+topic)
    data = json.loads(result.content)

    return {
        "target_audience": data["target_audience"],
        "primary_keyword": data["primary_keyword"],
        "search_intent": data["search_intent"],
        "desired_tone": data["recommended_tone"]
    }

def writer_node(state: ContentState):
    if state["iteration_count"] == 0:
        instruction = WRITER_PROMPT+f"""
        Write new article:
        Topic: {state['topic']}
        Target Audience: {state['target_audience']}
        Primary Keyword: {state['primary_keyword']}
        Search Intent: {state['search_intent']}
        Desired Tone: {state['desired_tone']}
        """
    else:
        instruction =WRITER_PROMPT + f"""
        Topic: {state['topic']}
        Target Audience: {state['target_audience']}
        Primary Keyword: {state['primary_keyword']}
        Search Intent: {state['search_intent']}
        Desired Tone: {state['desired_tone']}
        Revise the article using this feedback:
        SEO Feedback: {state['seo_evaluation']}
        Tone Feedback: {state['tone_evaluation']}
        """

    article = llm_with_tools.invoke(instruction)

    return {
        "article": article.content,
        "iteration_count": state["iteration_count"] + 1
    }

In [ ]:
import re
from datetime import datetime

filename = ""

def generate_filename(topic):
    safe_topic = re.sub(r'[^a-zA-Z0-9]+', '_', topic.lower())
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{safe_topic}_{timestamp}.md"

def seo_evaluator_node(state: ContentState):
    instruction = SEO_PROMPT + f"\n\nArticle: {state['article']}"
    result = llm.invoke(instruction)
    text = result.content.strip()

    if not text:
        raise ValueError("LLM returned empty output!")

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        raise ValueError(f"LLM output is not valid JSON:\n{text}")
        
    return {"seo_evaluation": data}

def tone_evaluator_node(state: ContentState):
    instruction = TONE_PROMPT + f"\n\nArticle: {state['article']}"
    result = llm.invoke(instruction)
    data = json.loads(result.content)
    return {"tone_evaluation": data}

def final_aggregator_node(state: ContentState):
    seo = state["seo_evaluation"]
    tone = state["tone_evaluation"]

    approved = seo["approved"] and tone["approved"]

    decision = {
        "final_decision": "publish" if approved else "revise",
        "needs_revision": not approved,
        "priority_fixes": list(
            set(seo["issues"][:2] + tone["issues"][:2])
        )
    }

    return {"final_decision": decision}

def save_to_file_node(state: ContentState):
    article = state["article"]
    output_dir = "outputs"
    os.makedirs(output_dir, exist_ok=True)
    global filename
    filename = generate_filename(state["topic"])
    output_path = os.path.join(output_dir, filename)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(article)

    instruction = f"""
    Send me a push notification to my phone with the following message:
    Article:
    {article}
    """
    result = llm_with_tools.invoke(instruction)
    

    return {"saved": True, "filename": filename}


def router_node(state: dict):
    if state["iteration_count"] >= state["max_iterations"]:
        return {"next_node": "save_to_file"}  # end of workflow

    if state["final_decision"]["needs_revision"]:
        return {"next_node": "writer"}

    return {"next_node": "save_to_file"} # <-- important




In [365]:

graph_builder = StateGraph(ContentState)

graph_builder.add_node("strategy_planner", strategy_planner_node)
graph_builder.add_node("writer", writer_node)
graph_builder.add_node("seo_evaluator", seo_evaluator_node)
graph_builder.add_node("tone_evaluator", tone_evaluator_node)
graph_builder.add_node("aggregator", final_aggregator_node)
graph_builder.add_node("save_to_file", save_to_file_node)
graph_builder.add_node("router", router_node)

graph_builder.set_entry_point("strategy_planner")

# Define edges
graph_builder.add_edge("strategy_planner", "writer")
graph_builder.add_edge("writer", "seo_evaluator")
graph_builder.add_edge("seo_evaluator", "tone_evaluator")
graph_builder.add_edge("tone_evaluator", "aggregator")
graph_builder.add_edge("aggregator", "router")
graph_builder.add_conditional_edges(
    "router",
    lambda state: state["next_node"],  # reads the decision from state
    {
        "writer": "writer",
        "save_to_file": "save_to_file"
    }
)
graph_builder.add_edge("save_to_file", END)

app = graph_builder.compile()

In [ ]:


def run_graph(topic: str):
    state = ContentState(
        topic=topic,
        target_audience="",
        primary_keyword="",
        search_intent="",
        desired_tone="",
        article=None,
        seo_evaluation=None,
        tone_evaluation=None,
        final_decision=None,
        iteration_count=0,
        max_iterations= SYSTEM_MAX_ITERATIONS,
        strict_mode_level=0,
    )
    app.invoke(state)
    global filename
    return state["article"], f"Saved to: {filename}"


In [368]:
with gr.Blocks() as demo:
    gr.Markdown("## Content Generator")
    topic_input = gr.Textbox(label="Enter Topic", placeholder="e.g. AI for Small Business Finance")
    article_output = gr.Markdown(label="Generated Article")
    status_output = gr.Textbox(label="Status / File Path", lines=2)
    generate_btn = gr.Button("Generate Article")

    generate_btn.click(
        fn=run_graph,
        inputs=topic_input,
        outputs=[article_output, status_output]
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/var/folders/ks/3lrl0nwx483f9zrxv2mmn01r0000gn/T/ipykernel_22152/1533218417.py", line 20, in seo_evaluator_node
    data = json.loads(text)
           ^^^^^^^^^^^^^^^^
  File "/Users/philmuhire/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/philmuhire/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/json/decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/philmuhire/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/json/decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)

During handling of the above exception, another 